# Trajectories of magnetic soft robot

### Imports

In [10]:
import numpy as np
import jax.numpy as jnp
import jax
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from softmobility import SoftBody, FlowBodySolver, Taylor_Green_flow, gravity_field, constant_scalar
from softmobility.classes.flowbodysolver import _rotation_matrix_from_Rodrigues_impl
from softmobility.classes.flowbodysolver import compute_Bortz_operator, rescale_orientation

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

In [11]:
BLUEACCENT="#8FC2CC" 
REDACCENT="#C04D27"

## Simulation of default parameters

### YAML file

In [12]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:
  - phi
design_names:      
  - spring_k
  - radius
input_names:
  - gravity
  - active_force
    
# Default Values (Optional)
defaults:
  spring_k: .5           
  radius: .33
  _spr_: 20
  _rad_: 1
  
# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: 1
    position: [0, 0, 1]                    
    orientation: [phi, 0, 0]
    force: [-gravity0, -gravity1, -gravity2]
    torque: [-_spr_ * spring_k * phi, 0, 0]              

  - # Sphere 1 #################
    radius: _rad_ * radius               
    position: [0, 0, -radius]
    orientation: [-phi/_rad_/radius, 0, 0]
    force: [gravity0, gravity1 + active_force * sin(phi/_rad_/radius), gravity2 + active_force * cos(phi/_rad_/radius)]
    torque: [_spr_ * spring_k * phi, 0, 0]
"""

In [13]:
yaml_data_rigid = """
# Variable Prefixes (Optional)
design_names:      
  - radius
input_names:
  - gravity
  - active_force
    
# Default Values (Optional)
defaults:
  radius: .5
  _rad_: 1
  
# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: 1
    position: [0, 0, 1]                    
    orientation: [0, 0, 0]
    force: [-gravity0, -gravity1, -gravity2]
    torque: [0, 0, 0]              

  - # Sphere 1 #################
    radius: _rad_ * radius               
    position: [0, 0, -radius]
    orientation: [0, 0, 0]
    force: [gravity0, gravity1, gravity2 + active_force]
    torque: [0, 0, 0]
"""

### Soft Body

In [14]:
mybody= SoftBody(yaml_data, verbose=True)
myrigidbody = SoftBody(yaml_data_rigid, verbose=False)

  Found variables: phi, radius, spring_k, gravity0, gravity1, gravity2, active_force
  3D field inputs:  ['gravity']
  Scalar inputs:    ['active_force']
    Sphere 0
      Radius: 1
      Position: ['0', '0', '1']
      Orientation: ['phi', '0', '0']
      Coupling matrix C_H:
[['-1', '0', '0', '0'], ['0', '-1', '0', '0'], ['0', '0', '-1', '0'], ['0', '0', '0', '0'], ['0', '0', '0', '0'], ['0', '0', '0', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['-20*spring_k'], ['0'], ['0']]
    Sphere 1
      Radius: radius
      Position: ['0', '0', '-radius']
      Orientation: ['-phi/radius', '0', '0']
      Coupling matrix C_H:
[['1', '0', '0', '0'], ['0', '1', '0', 'sin(phi/radius)'], ['0', '0', '1', 'cos(phi/radius)'], ['0', '0', '0', '0'], ['0', '0', '0', '0'], ['0', '0', '0', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['20*spring_k'], ['0'], ['0']]


### Parameters of flow and gravity field

In [15]:
# Calculating force to have a swimming velocity of 1
tensors = myrigidbody.compute_fast_mobility(dofs=jnp.zeros(myrigidbody.Ndof))
FORCE_INTENSITY=1/tensors.M_H[2,-1]

In [16]:
# Create a TGV flow with with max vorticy = 1
FLOW = Taylor_Green_flow(omega=2)
# Create a gravity field 
GRAVITY = gravity_field(g=50.0)
# Create an active force scalar
FORCE = constant_scalar(value=FORCE_INTENSITY)

In [29]:
# parameters of time integration
FINAL_TIME = 1 * jnp.pi  
DT = 0.3
N_TRAJECTORIES = 1

Y_INIT_POSITIONS = np.linspace(np.pi/N_TRAJECTORIES/2, 2*np.pi, N_TRAJECTORIES, endpoint=False)
N_DT = int(FINAL_TIME / DT)


### Brute-force optimization of stiffness

In [9]:
spring_ks = np.linspace(0, 1, 11)  # [0.0, 0.1, ..., 1.0]
results = []

for spring_k in spring_ks:
    mybody.set_design_defaults(new_dict={'spring_k': spring_k})

    trajectories = []
    for x_init in Y_INIT_POSITIONS:
        solver = FlowBodySolver(
            soft_body=mybody,
            flow=FLOW,
            input_map={"gravity": GRAVITY, "active_force": FORCE},
            dt=DT,
            init_position=[np.pi/2, x_init, 0],
            init_orientation=[0, 0, 0],
            integrator="RK2")
        trajectories.append(solver.simulate(final_time=FINAL_TIME))

    end_z_positions = [float(jnp.array([t[0] for t in traj])[-1, 2]) for traj in trajectories]
    mean_velocity = np.mean(end_z_positions) / FINAL_TIME
    results.append((spring_k, mean_velocity))
    print(f"spring_k={spring_k:.2f}  →  mean velocity={mean_velocity:.4f}")

# --- Summary ---
best_k, best_v = max(results, key=lambda x: x[1])
print(f"\nBest spring_k={best_k:.2f}  →  mean velocity={best_v:.4f}")

Time: 0.000 / 12.566  Integrator RK2
Time: 0.000 / 12.566  Integrator RK2
Time: 0.000 / 12.566  Integrator RK2
spring_k=0.10  →  mean velocity=0.1780
OLD default param values: ['radius', 'spring_k'] [ 0.33  0.1 ]
NEW default param values: ['radius', 'spring_k'] [ 0.33  0.2 ]
Time: 0.000 / 12.566  Integrator RK2


KeyboardInterrupt: 

### Functions

In [18]:
rotation_matrix_from_Rodrigues = jax.jit(
    lambda rvec, Ndof: _rotation_matrix_from_Rodrigues_impl(rvec, Ndof), static_argnums=(1,)
)

In [ ]:
def sixc_velocity(design, orientation, dofs, field_lab, scalar, u_lab, omega_lab, E_lab):
    rot_matrix, sixc_rot_matrix = rotation_matrix_from_Rodrigues(orientation, Ndof=mybody.Ndof)
    
    E_body = rot_matrix.T @ E_lab @ rot_matrix
    E_inf = jnp.array([E_body[0, 0], E_body[0, 1], E_body[0, 2], E_body[1, 1], E_body[1, 2]])

    field_body = rot_matrix.T @ field_lab
    input_vec = jnp.concatenate([field_body, jnp.array([scalar])])
    
    tensors = mybody.compute_mobility_problem(dofs, design)
    sixc_velocity = tensors.M_H @ input_vec + tensors.M_K @ dofs + tensors.C_E @ E_inf

    p_lab = jnp.block([u_lab, omega_lab, jnp.zeros(mybody.Ndof)])
    return sixc_rot_matrix @ sixc_velocity + p_lab

In [38]:
def _rollout_one(design, x_init):
    """Simulate one trajectory and return mean Z velocity."""
    position    = jnp.array([np.pi/2, x_init, 0.0])
    orientation = jnp.zeros(3)
    dofs        = jnp.ones(mybody.Ndof) * 1e-6   # jnp.zeros causes a singularity


    def step(carry, t):
        position, orientation, dofs = carry
        time = t * DT

        field_lab = GRAVITY.vector(position, time)
        scalar    = FORCE.value(position, time)
        u_lab     = FLOW.velocity(position, time)
        omega_lab, E_lab = FLOW.omega_rate_of_strain(position, time)
        bortz     = compute_Bortz_operator(orientation)

        # RK2
        p1          = sixc_velocity(design, orientation, dofs, field_lab, scalar, u_lab, omega_lab, E_lab)
        k1_pos, k1_ori, k1_dof = p1[:3], p1[3:6], p1[6:]

        pos_mid = position    + DT * k1_pos / 2
        ori_mid = orientation + DT * bortz @ k1_ori / 2
        dof_mid = dofs        + DT * k1_dof / 2

        field_lab2  = GRAVITY.vector(pos_mid, time + DT/2)
        scalar2    = FORCE.value(pos_mid, time + DT/2)
        u_lab2     = FLOW.velocity(pos_mid, time + DT/2)
        omega_lab2, E_lab2 = FLOW.omega_rate_of_strain(pos_mid, time + DT/2)
        p2          = sixc_velocity(design, ori_mid, dof_mid, field_lab2, scalar2, u_lab2, omega_lab2, E_lab2)

        k2_pos, k2_ori, k2_dof = p2[:3], p2[3:6], p2[6:]

        position    = position    + DT * (k1_pos + k2_pos) / 2
        orientation = orientation + DT * bortz @ (k1_ori + k2_ori) / 2
        orientation = rescale_orientation(orientation)
        dofs        = dofs        + DT * (k1_dof + k2_dof) / 2

        return (position, orientation, dofs), None

    (position, orientation, dofs), _ = jax.lax.scan(
        step, (position, orientation, dofs), jnp.arange(N_DT)
    )

    return position[2] / FINAL_TIME   # mean Z velocity


In [39]:
def mean_velocity_objective(design):
    """Negative mean Z velocity across all initial positions (negative = minimize to maximize)."""
    velocities = jnp.stack([_rollout_one(design, x_init) for x_init in Y_INIT_POSITIONS])
    return -jnp.mean(velocities)


grad_fn = jax.jit(jax.value_and_grad(mean_velocity_objective))

In [44]:
def gradient_descent(
    grad_fn,
    initial_design,
    lr          = 0.05,
    n_steps     = 1000,
    print_every = 100,
    clip_min    = 0.0,
    clip_max    = 1.0,
):
    design = initial_design
    design_keys = mybody.design_variables
    loss, grad = None, None

    for step in range(n_steps):
        loss, grad = grad_fn(design)
        design     = design - lr * grad
        design     = jnp.clip(design, clip_min, clip_max)

        if step % print_every == 0 or step == n_steps - 1:
            params_str = "  ".join(f"{k}={float(v):.4f}" for k, v in zip(design_keys, design))
            print(f"step {step:4d}  velocity={float(-loss):.5f}  |grad|={float(jnp.linalg.norm(grad)):.5f}  {params_str}")

    print(f"\nOptimal design:")
    for k, v in zip(design_keys, design):
        print(f"  {k}: {float(v):.4f}  (init: {float(initial_design[design_keys.index(k)]):.4f})")
    print(f"Mean velocity: {float(-loss):.5f}")

    return design, float(-loss)

In [ ]:

# --- Run ---
initial_design = 0.5 * jnp.ones(mybody.Ndesign)

optimal_design, optimal_velocity = gradient_descent(
    grad_fn,
    initial_design,
    lr          = 0.01,
    n_steps     = 1000,
    print_every = 100,
)

step    0  velocity=1.17877  |grad|=0.14655  radius=0.4931  spring_k=0.5024
step  100  velocity=1.20729  |grad|=0.06297  radius=0.3776  spring_k=0.8473
step  200  velocity=1.21647  |grad|=0.05242  radius=0.3361  spring_k=1.0000
step  300  velocity=1.21647  |grad|=0.05246  radius=0.3360  spring_k=1.0000
step  400  velocity=1.21647  |grad|=0.05246  radius=0.3360  spring_k=1.0000
step  500  velocity=1.21647  |grad|=0.05246  radius=0.3360  spring_k=1.0000
step  600  velocity=1.21647  |grad|=0.05246  radius=0.3360  spring_k=1.0000
step  700  velocity=1.21647  |grad|=0.05246  radius=0.3360  spring_k=1.0000
step  800  velocity=1.21647  |grad|=0.05246  radius=0.3360  spring_k=1.0000
step  900  velocity=1.21647  |grad|=0.05246  radius=0.3360  spring_k=1.0000
step  999  velocity=1.21647  |grad|=0.05246  radius=0.3360  spring_k=1.0000

Optimal design:
  radius: 0.3360  (init: 0.5000)
  spring_k: 1.0000  (init: 0.5000)
Mean velocity: 1.21647


In [41]:
# --- Gradient descent ---
design   = 0.5 * jnp.ones(mybody.Ndesign)   
lr       = 0.05
n_steps  = 30

for step in range(n_steps):
    loss, grad = grad_fn(design)
    design     = design - lr * grad
    design     = jnp.clip(design, 0.0, 1.0)   # keep spring_k in valid range
    print(f"step {step:3d}  design={design}  mean_velocity={float(-loss):.5f}  grad={grad}")

print(f"\nOptimal design={design}  →  mean velocity={float(-loss):.5f}")

step   0  design=[ 0.493  0.502]  mean_velocity=1.17877  grad=[ 0.138 -0.049]
step   1  design=[ 0.487  0.505]  mean_velocity=1.17977  grad=[ 0.116 -0.054]
step   2  design=[ 0.482  0.508]  mean_velocity=1.18054  grad=[ 0.098 -0.058]
step   3  design=[ 0.478  0.511]  mean_velocity=1.18115  grad=[ 0.083 -0.061]
step   4  design=[ 0.475  0.514]  mean_velocity=1.18166  grad=[ 0.071 -0.064]
step   5  design=[ 0.472  0.518]  mean_velocity=1.18211  grad=[ 0.061 -0.066]
step   6  design=[ 0.469  0.521]  mean_velocity=1.18250  grad=[ 0.053 -0.068]
step   7  design=[ 0.467  0.524]  mean_velocity=1.18287  grad=[ 0.047 -0.07 ]
step   8  design=[ 0.465  0.528]  mean_velocity=1.18322  grad=[ 0.042 -0.071]
step   9  design=[ 0.463  0.532]  mean_velocity=1.18356  grad=[ 0.038 -0.072]
step  10  design=[ 0.461  0.535]  mean_velocity=1.18388  grad=[ 0.034 -0.073]
step  11  design=[ 0.459  0.539]  mean_velocity=1.18421  grad=[ 0.031 -0.073]
step  12  design=[ 0.458  0.543]  mean_velocity=1.18452  grad=[ 

### Simulations for plots

In [ ]:
mybody.set_design_defaults(new_dict={
        'spring_k': best_k, 
})

trajectories = []

for x_init in Y_INIT_POSITIONS:
    print("New initial position", x_init)
    solver = FlowBodySolver(
        soft_body=mybody, 
        flow=FLOW, 
        input_map={"gravity": GRAVITY, "active_force": FORCE}, 
        dt=DT, 
        init_position=[np.pi/2, x_init, 0],
        init_orientation=[0, 0, 0], 
        integrator="RK2")
    trajectories.append(solver.simulate(final_time=FINAL_TIME))
    
end_z_positions = [float(jnp.array([t[0] for t in traj])[-1, 2]) for traj in trajectories]
mean_end_z = np.mean(end_z_positions)

print(f"End Z positions: {[f'{z/np.pi:.3g}π' for z in end_z_positions]}")
print(f"Mean velocity: {mean_end_z/FINAL_TIME:.3g}")

In [ ]:
rigid_trajectories = []

for x_init in Y_INIT_POSITIONS:
    print("New initial position", x_init)
    solver = FlowBodySolver(
        soft_body=myrigidbody, 
        flow=FLOW, 
        input_map={"gravity": GRAVITY, "active_force": FORCE}, 
        dt=DT, 
        init_position=[np.pi/2, x_init, 0],
        init_orientation=[0, 0, 0], 
        integrator="RK2")
    rigid_trajectories.append(solver.simulate(final_time=FINAL_TIME))
    
end_z_positions = [float(jnp.array([t[0] for t in traj])[-1, 2]) for traj in rigid_trajectories]
mean_end_z = np.mean(end_z_positions)

print(f"End Z positions: {[f'{z/np.pi:.3g}π' for z in end_z_positions]}")
print(f"Mean velocity: {mean_end_z/FINAL_TIME:.3g}")

### Preparing the figure of trajectory

In [ ]:
# --- Helper to build tick labels like "−2π", "−π/2", "0", "π", etc. ---
def pi_ticks(lo, hi, step_pi=1):
    """Return (tickvals, ticktext) with spacing of step_pi * π."""
    import math
    n_lo = math.floor(lo / (step_pi * np.pi))
    n_hi = math.ceil(hi  / (step_pi * np.pi))
    vals, texts = [], []
    for n in range(n_lo, n_hi + 1):
        v = n * step_pi * np.pi
        vals.append(v)
        k = n * step_pi          # multiple of π
        if   k == 0:  texts.append('0')
        elif k == 1:  texts.append('π')
        elif k == -1: texts.append('−π')
        else:         texts.append(f'{k}π')
    return vals, texts

def bounds_2pi(lo, hi, step_pi=2):
    """Round lo down and hi up to the nearest multiple of step_pi*π."""
    unit = step_pi * np.pi
    return np.floor(lo / unit) * unit, np.ceil(hi / unit) * unit

In [ ]:
colors=[REDACCENT, BLUEACCENT]

# --- Compute global bounds across ALL trajectories ---
all_y = np.concatenate([np.array(jnp.array([t[0] for t in traj])[:, 1]) for traj in trajectories + rigid_trajectories])
all_z = np.concatenate([np.array(jnp.array([t[0] for t in traj])[:, 2]) for traj in trajectories + rigid_trajectories])

Y_MIN, Y_MAX = bounds_2pi(all_y.min(), all_y.max())
Z_MIN, Z_MAX = bounds_2pi(all_z.min(), all_z.max())

GRID_STEP = 1
y_ticks, _ = pi_ticks(Y_MIN, Y_MAX, step_pi=GRID_STEP)
z_ticks, _ = pi_ticks(Z_MIN, Z_MAX, step_pi=GRID_STEP)

ASPECT     = (Y_MAX - Y_MIN)
FIG_WIDTH  = 800   # total width for both panels
FIG_HEIGHT = int((FIG_WIDTH / 2) * (Z_MAX - Z_MIN) / ASPECT)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Soft", "Rigid"),
                    horizontal_spacing=0.05)

# --- Helper to add one set of trajectories to a given col ---
def add_trajectories(fig, traj_list, col):
    for i, (trajectory, x_init) in enumerate(zip(traj_list, Y_INIT_POSITIONS)):
        positions = jnp.array([t[0] for t in trajectory])
        y_pos = np.array(positions[:, 1])
        z_pos = np.array(positions[:, 2])
        label = f"y₀={x_init/np.pi:.2g}π"

        fig.add_trace(go.Scatter(
            x=y_pos, y=z_pos,
            mode='lines',
            line=dict(color=colors[col-1], width=2),
            name=label, showlegend=False,
            hovertemplate=f"Y=%{{x:.3f}}<br>Z=%{{y:.3f}}<extra>{label}</extra>",
        ), row=1, col=col)
        
        fig.add_trace(go.Scatter(
            x=[y_pos[-1]], y=[z_pos[-1]],
            mode='markers',
            marker=dict(color=colors[col-1], size=8, symbol=['circle', 'square']),
            showlegend=False,
            hovertemplate=f"%{{text}}<extra>{label}</extra>",
            text=['start', 'end'],
        ), row=1, col=col)

add_trajectories(fig, trajectories,       col=1)
add_trajectories(fig, rigid_trajectories, col=2)

# --- Shared axis style ---
axis_style = dict(
    showticklabels=False,
    showgrid=True, gridcolor='lightgrey', gridwidth=1,
    zeroline=True, linecolor='lightgrey', mirror=True,
    title=None,
    constrain='domain',
)

fig.update_layout(
    template='plotly_white',
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    margin=dict(l=20, r=20, t=30, b=20),
    showlegend=False,
    xaxis=dict(**axis_style, range=[Y_MIN, Y_MAX], scaleanchor='y',  scaleratio=1, tickvals=y_ticks),
    yaxis=dict(**axis_style, range=[Z_MIN, Z_MAX], constraintoward='center', tickvals=z_ticks),
    xaxis2=dict(**axis_style, range=[Y_MIN, Y_MAX], scaleanchor='y2', scaleratio=1, tickvals=y_ticks),
    yaxis2=dict(**axis_style, range=[Z_MIN, Z_MAX], constraintoward='center', tickvals=z_ticks),
)


### Figure

In [ ]:
fig.show()